# 01 — Ingest & Transform

Loads the raw synthetic finance-ops CSVs (`data/raw/`) and produces a cleaned, analysis-ready table in `data/processed/`.

**Scope of this notebook:** standardize schema and fix data-quality mess (inconsistent text casing/whitespace, mixed date-string formats, a handful of missing timestamps). It does **not** compute KPIs or detect anomalies — that's `src/kpis.py` and `src/anomalies.py` (stage 3), which read the *cleaned* output this notebook writes.

**Local vs. Databricks:** the same notebook runs unmodified in both places. The first code cell detects Databricks and switches paths: locally it reads/writes `data/raw` and `data/processed`; on Databricks (Free Edition) it reads the CSVs from a Unity Catalog **Volume** (`/Volumes/workspace/finance_ops/files/raw`). All cleaning is plain pandas either way — at ~2,500 rows, distributed processing isn't justified.

The one Databricks-only addition is at the end: the cleaned pandas DataFrames are registered as **Delta tables** in Unity Catalog (`workspace.finance_ops.*`) and cross-checked with a Databricks SQL query. Spark is used there only as the write/query interface to the catalog — no transformation logic runs in Spark.

**Extended data:** supplier payment terms are parsed into discount and net days, PO numbers are normalised (a missing PO is *maverick spend*), invoice due and discount dates are derived, and the process event log is checked against the invoice dates. The extract date comes from `extract_metadata.json`: invoices not yet paid by then are open items.


In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Databricks sets this env var on every cluster/serverless runtime.
ON_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# Unity Catalog location (Databricks only): catalog.schema, plus a Volume for files.
CATALOG_SCHEMA = 'workspace.finance_ops'
VOLUME_DIR = Path('/Volumes/workspace/finance_ops/files')

if ON_DATABRICKS:
    RAW_DIR = VOLUME_DIR / 'raw'
    PROCESSED_DIR = VOLUME_DIR / 'processed'
else:
    RAW_DIR = Path('..') / 'data' / 'raw'
    PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'running on Databricks: {ON_DATABRICKS} | pandas {pd.__version__} | raw data from {RAW_DIR}')
pd.set_option('display.max_columns', None)

running on Databricks: False | pandas 3.0.6 | raw data from ../data/raw


In [2]:
invoices_raw = pd.read_csv(RAW_DIR / 'invoices.csv', dtype=str)
vendors_raw = pd.read_csv(RAW_DIR / 'vendors.csv', dtype=str)
budgets = pd.read_csv(RAW_DIR / 'budgets.csv')
events_raw = pd.read_csv(RAW_DIR / 'invoice_events.csv', dtype=str)
EXTRACT_DATE = pd.Timestamp(json.loads((RAW_DIR / 'extract_metadata.json').read_text())['extract_date'])

print(f'invoices: {len(invoices_raw)} rows')
print(f'vendors:  {len(vendors_raw)} rows')
print(f'budgets:  {len(budgets)} rows')
print(f'events:   {len(events_raw)} rows')
print(f'extract date: {EXTRACT_DATE.date()}')
invoices_raw.head(3)

invoices: 2550 rows
vendors:  40 rows
budgets:  432 rows
events:   10882 rows
extract date: 2026-08-31


,invoice_id,vendor_id,po_number,department,category,invoice_date,submitted_date,approved_date,paid_date,amount,currency,status
0,INV00001,V020,-,Legal,Software & IT Services,2026-01-19,2026-01-23,2026-01-26,2026-01-29,2122.75,EUR,Paid
1,INV00002,V003,4560641662,Operations,Consulting,2026-07-17,2026-07-18,2026-07-22,2026-07-23,760.84,EUR,Paid
2,INV00003,V016,4573723628,Marketing,Travel,2026-04-13,2026-04-13,2026-04-21,2026-04-26,19385.8,EUR,Paid


## Explore the mess

Before cleaning anything, look at what's actually wrong. This is the step that turns "clean this data" from a guess into a checklist.

In [3]:
print('department value counts:')
print(invoices_raw['department'].value_counts())
print()
print('category value counts:')
print(invoices_raw['category'].value_counts())
print()
print('status value counts:')
print(invoices_raw['status'].value_counts())
print()

iso_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
non_iso_dates = (~invoices_raw['invoice_date'].str.match(iso_pattern)).sum()
print(f'invoice_date values not in ISO format: {non_iso_dates}')
print(f'missing submitted_date: {invoices_raw["submitted_date"].isna().sum()}')
print()
print('payment_terms value counts:')
print(vendors_raw['payment_terms'].value_counts())
print()
# Digits shown as 9, so the patterns are visible rather than individual numbers.
print('po_number patterns:')
print(invoices_raw['po_number'].fillna('<empty>').str.replace(r'\d', '9', regex=True).value_counts())

department value counts:
department
Operations        398
Legal             391
HR                368
IT                364
Marketing         362
Finance           362
  Legal            27
Hr                 23
MARKETING          21
finance            20
marketing          18
OPERATIONS         18
It                 18
it                 18
legal              17
  Marketing        17
LEGAL              16
FINANCE            15
operations         15
  HR               15
  Finance          13
  IT               12
hr                 12
  Operations       10
Name: count, dtype: int64

category value counts:
category
Professional Services         506
Consulting                    495
Software & IT Services        452
Office Supplies               374
Travel                        244
Facilities                    179
  Consulting                   27
  Professional Services        27
PROFESSIONAL SERVICES          23
  Software & IT Services       23
SOFTWARE & IT SERVICES         23
CON

## Standardize categorical text

`department`/`category`/`status` carry the same value in different casings and with stray whitespace (`" it "`, `"IT"`, `"It"`). Blindly calling `.str.title()` would break an abbreviation like `"IT"` (title-cases to `"It"`). Instead we **casefold + strip, then map through a lookup table** built from the known canonical values — a more robust pattern for any fixed-vocabulary field, not just this dataset.

`status` has four canonical values: `Paid`, and three open states at the extract date (`Approved`, `Pending approval`, `Received`).


In [4]:
DEPARTMENTS = ['Finance', 'IT', 'Marketing', 'Operations', 'HR', 'Legal']
CATEGORIES = ['Consulting', 'Software & IT Services', 'Facilities', 'Travel', 'Office Supplies', 'Professional Services']
STATUSES = ['Paid', 'Approved', 'Pending approval', 'Received']

def build_canon_map(canonical_values):
    return {v.strip().lower(): v for v in canonical_values}

def normalize_categorical(series: pd.Series, canon_map: dict) -> pd.Series:
    cleaned = series.str.strip().str.lower().map(canon_map)
    unmatched = cleaned.isna().sum()
    if unmatched:
        print(f'  warning: {unmatched} values in {series.name!r} did not match the canonical map')
    return cleaned

invoices = invoices_raw.copy()
invoices['department'] = normalize_categorical(invoices['department'], build_canon_map(DEPARTMENTS))
invoices['category'] = normalize_categorical(invoices['category'], build_canon_map(CATEGORIES))
invoices['status'] = normalize_categorical(invoices['status'], build_canon_map(STATUSES))

assert invoices['department'].isin(DEPARTMENTS).all()
assert invoices['category'].isin(CATEGORIES).all()
assert invoices['status'].isin(STATUSES).all()
invoices[['department', 'category', 'status']].nunique()

department    6
category      6
status        4
dtype: int64

## Clean vendor names

`vendor_name` is free text (not a fixed vocabulary), so there's no lookup table — just strip leading/trailing whitespace and collapse repeated internal spaces.

In [5]:
vendors = vendors_raw.copy()
vendors['vendor_name'] = vendors['vendor_name'].str.strip().str.replace(r'\s+', ' ', regex=True)
vendors.head(3)

,vendor_id,vendor_name,category,payment_terms,preferred_vendor
0,V001,"Rodriguez, Figueroa and Sanchez",Professional Services,2/10 Net 30,Y
1,V002,Doyle Ltd,Consulting,2/10 Net 30,Y
2,V003,"Mcclain, Miller and Henderson",Consulting,Net 30,N


## Parse payment terms

`payment_terms` is free text on the supplier master, typed several ways (`2/10 Net 30`, `2/10 NET30`, ` net 45 `). One regular expression turns each into three numbers: discount %, discount days and net days. `2/10 Net 30` means 2% off if paid within 10 days of the invoice date, otherwise due in 30 days. A value the pattern doesn't match fails loudly instead of becoming a silent NaN.

`preferred_vendor` (`Y`/`N`) becomes a boolean.


In [6]:
TERMS_PATTERN = re.compile(r'^(?:(\d+(?:\.\d+)?)/(\d+)\s*)?net\s*(\d+)$')

def parse_payment_terms(terms: str) -> dict:
    match = TERMS_PATTERN.match(terms.strip().lower())
    assert match, f'unrecognised payment terms: {terms!r}'
    pct, discount_days, net_days = match.groups()
    return {
        'discount_pct': float(pct) if pct else 0.0,
        'discount_days': int(discount_days) if discount_days else None,
        'net_days': int(net_days),
    }

parsed = pd.DataFrame([parse_payment_terms(t) for t in vendors['payment_terms']], index=vendors.index)
vendors['discount_pct'] = parsed['discount_pct']
vendors['discount_days'] = parsed['discount_days'].astype('Int64')
vendors['net_days'] = parsed['net_days'].astype('Int64')
# Rewrite in one canonical spelling, for display.
vendors['payment_terms'] = [
    f"{pct:g}/{dd} Net {nd}" if pd.notna(dd) else f"Net {nd}"
    for pct, dd, nd in zip(vendors['discount_pct'], vendors['discount_days'], vendors['net_days'])
]

vendors['preferred_vendor'] = vendors['preferred_vendor'].str.strip().str.upper().map({'Y': True, 'N': False})
assert vendors['preferred_vendor'].notna().all(), 'preferred_vendor must be Y or N'

vendors['payment_terms'].value_counts()

payment_terms
Net 30         19
2/10 Net 30    10
Net 14          6
Net 45          5
Name: count, dtype: int64

## Normalise PO numbers

An invoice without a purchase order is **maverick spend**, so a missing PO must be recognised however it was typed: empty, `N/A`, `-`, `none`. Valid numbers sometimes carry spaces or a `PO ` prefix. The rule: take the 10-digit number starting with 45 (the ERP's purchase-order number range); anything else means no PO. `has_po` makes that explicit.


In [7]:
invoices['po_number'] = invoices_raw['po_number'].str.extract(r'(45\d{8})', expand=False)
invoices['has_po'] = invoices['po_number'].notna()

# Every raw value that was not a PO number must be a known way of writing "none".
treated_as_missing = invoices_raw['po_number'].notna() & invoices['po_number'].isna()
leftover = set(invoices_raw.loc[treated_as_missing, 'po_number'].str.strip().str.lower())
assert leftover <= {'-', 'none', ''}, f'unexpected po_number values: {leftover}'

print(f"invoices with a PO: {invoices['has_po'].sum()} of {len(invoices)}")

invoices with a PO: 2033 of 2550


## Parse mixed date formats

`invoice_date` shows up in three different string formats (`YYYY-MM-DD`, `MM/DD/YYYY`, `DD-Mon-YYYY`, `YYYY/MM/DD`) — a realistic symptom of data entered by different people/systems. `pandas.to_datetime` can't infer a single consistent format across all of them reliably, so we try a list of known formats per value instead of relying on format inference.

In [8]:
KNOWN_DATE_FORMATS = ['%Y-%m-%d', '%m/%d/%Y', '%d-%b-%Y', '%Y/%m/%d']

def parse_mixed_dates(series: pd.Series) -> pd.Series:
    parsed = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    remaining = series.notna()
    for fmt in KNOWN_DATE_FORMATS:
        if not remaining.any():
            break
        attempt = pd.to_datetime(series[remaining], format=fmt, errors='coerce')
        matched = attempt.notna()
        parsed.loc[attempt.index[matched]] = attempt[matched]
        remaining.loc[attempt.index[matched]] = False
    return parsed

invoices['invoice_date'] = parse_mixed_dates(invoices['invoice_date'])
invoices['submitted_date'] = parse_mixed_dates(invoices['submitted_date'])
invoices['approved_date'] = parse_mixed_dates(invoices['approved_date'])
invoices['paid_date'] = parse_mixed_dates(invoices['paid_date'])

unparsed = invoices['invoice_date'].isna().sum()
print(f'invoice_date rows that failed to parse: {unparsed} (should be 0)')
invoices[['invoice_date', 'submitted_date', 'approved_date', 'paid_date']].dtypes

invoice_date rows that failed to parse: 0 (should be 0)


invoice_date      datetime64[ns]
submitted_date    datetime64[ns]
approved_date     datetime64[ns]
paid_date         datetime64[ns]
dtype: object

## Handle missing `submitted_date`

A few rows (~1%) are missing `submitted_date` entirely — a plausible real-world gap (a workflow step that didn't log a timestamp). We don't drop these rows outright: `amount`/`department`/`category` are still valid, so they should still count toward spend KPIs. We only flag them so the processing-time KPI (which needs `submitted_date`) can exclude them explicitly instead of silently mis-computing on a NaT.

An invoice with status `Received` was not yet submitted at the extract date, so its empty `submitted_date` is expected, not a data gap, and is not flagged. Processing time also needs a `paid_date`, so it is empty for open invoices too.


In [9]:
invoices['missing_submitted_date'] = invoices['submitted_date'].isna() & (invoices['status'] != 'Received')
invoices['processing_days'] = (invoices['paid_date'] - invoices['submitted_date']).dt.days

invoices['amount'] = pd.to_numeric(invoices['amount'])
# Carried on every row, so downstream code can age open invoices without a separate config value.
invoices['extract_date'] = EXTRACT_DATE

print(f"flagged missing_submitted_date: {invoices['missing_submitted_date'].sum()} rows")
print(f"open at extract (no paid_date): {invoices['paid_date'].isna().sum()} rows")
print(f"processing_days is null for: {invoices['processing_days'].isna().sum()} rows (missing submission or still open)")

flagged missing_submitted_date: 24 rows
open at extract (no paid_date): 93 rows
processing_days is null for: 117 rows (missing submission or still open)


## Derive due and discount dates

The extract has no due date: the ERP computes it from a baseline date and the payment terms. Here the baseline is the invoice date. `due_date` = invoice date + net days. `discount_date` = invoice date + discount days, only for suppliers with discount terms. These two dates drive the on-time payment rate, missed discounts and late-payment cost in Measure.


In [10]:
invoices = invoices.merge(
    vendors[['vendor_id', 'payment_terms', 'discount_pct', 'discount_days', 'net_days']],
    on='vendor_id', how='left', validate='many_to_one',
)
invoices['due_date'] = invoices['invoice_date'] + pd.to_timedelta(invoices['net_days'], unit='D')
invoices['discount_date'] = invoices['invoice_date'] + pd.to_timedelta(invoices['discount_days'], unit='D')

print(f"invoices with discount terms: {invoices['discount_date'].notna().sum()}")
invoices[['invoice_id', 'invoice_date', 'payment_terms', 'discount_date', 'due_date']].head(3)

invoices with discount terms: 630


,invoice_id,invoice_date,payment_terms,discount_date,due_date
0,INV00001,2026-01-19,Net 30,NaT,2026-02-18
1,INV00002,2026-07-17,Net 30,NaT,2026-08-16
2,INV00003,2026-04-13,2/10 Net 30,2026-04-23,2026-05-13


## Validate the process event log

`invoice_events.csv` has one row per process step (receive, submit, reject, resubmit, approve, pay), with the **role** that did it but never a person. It feeds process mining in Analyze. Before using it, check that it agrees with the invoice table: every case is a known invoice, nothing happens after the extract, and each step's date equals the matching date column.


In [11]:
events = events_raw.copy()
events['timestamp'] = pd.to_datetime(events['timestamp'], format='%Y-%m-%dT%H:%M:%S')
events = events.sort_values(['invoice_id', 'timestamp'], kind='stable').reset_index(drop=True)

assert events['invoice_id'].isin(invoices['invoice_id']).all(), 'every event must belong to a known invoice'
assert (events['timestamp'].dt.normalize() <= EXTRACT_DATE).all(), 'no event may happen after the extract'

STEP_DATE_COLUMNS = {
    'Receive invoice': 'invoice_date',
    'Submit for approval': 'submitted_date',
    'Approve invoice': 'approved_date',
    'Pay invoice': 'paid_date',
}
by_invoice = invoices.set_index('invoice_id')
for activity, column in STEP_DATE_COLUMNS.items():
    step_dates = events[events['activity'] == activity].set_index('invoice_id')['timestamp'].dt.normalize()
    expected = by_invoice[column].dropna()
    assert step_dates.index.is_unique, f'{activity} must happen at most once per invoice'
    assert set(step_dates.index) == set(expected.index), f'{activity} events must exist exactly where {column} is set'
    assert (step_dates[expected.index] == expected).all(), f'{activity} dates must equal {column}'

print(events['activity'].value_counts())

activity
Receive invoice         2550
Submit for approval     2505
Approve invoice         2482
Pay invoice             2457
Reject invoice           446
Correct and resubmit     442
Name: count, dtype: int64


## Validate before writing

Cheap assertions that would catch a broken cleaning step before it silently corrupts stage 3 (KPIs/anomaly detection).

In [12]:
assert invoices['invoice_id'].is_unique, 'invoice_id must stay unique — planted duplicates use a new id, not a repeated one'
assert len(invoices) == len(invoices_raw), 'row count must be preserved — this notebook cleans, it does not drop/dedupe'
assert invoices['amount'].gt(0).all(), 'amount must stay positive after cleaning'
assert invoices['department'].isin(DEPARTMENTS).all()
assert invoices['category'].isin(CATEGORIES).all()
assert invoices['status'].eq('Paid').eq(invoices['paid_date'].notna()).all(), 'an invoice is Paid exactly when it has a paid_date'
for col in ['invoice_date', 'submitted_date', 'approved_date', 'paid_date']:
    assert (invoices[col].dropna() <= EXTRACT_DATE).all(), f'{col} must not be after the extract date'
assert invoices['net_days'].notna().all(), 'every invoice must get payment terms from its vendor'
assert (invoices['due_date'] >= invoices['invoice_date']).all()
assert invoices['discount_date'].isna().eq(invoices['discount_days'].isna()).all()

print('all validation checks passed')

all validation checks passed


## Log what cleaning repaired

Cleaning fixes the data here, but the cause stays at the source: the same misspelt department or `NET30` will
arrive in next month's extract. So the notebook records each kind of repair, with how many rows it touched and
a few of the raw spellings. The app's data-quality view reads this log from `data/processed/`, so it never has
to look at the raw files or repeat the cleaning rules.

One trap: `read_csv` turns the text `N/A` and `n/a` into a missing value on its own, so in `invoices_raw` they
look like empty fields. The PO count therefore re-reads that one column exactly as typed.

In [13]:
def raw_examples(raw: pd.Series, changed: pd.Series, n: int = 3) -> str:
    """One raw value for each of the n most common shapes among the changed rows,
    quoted so stray spaces stay visible. Digits count as the same shape, so dates
    show one example per format instead of three dates in the same format.
    """
    values = raw[changed]
    shapes = values.str.replace(r'\d', '9', regex=True)
    counts = shapes.value_counts()
    top = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:n]
    return ' · '.join(repr(values[shapes == shape].min()) for shape, _ in top)

po_raw = invoices_raw['po_number']
# As typed: without this, read_csv has already turned 'N/A' and 'n/a' into empty values.
po_as_typed = pd.read_csv(RAW_DIR / 'invoices.csv', dtype=str, keep_default_na=False, usecols=['po_number'])['po_number']
po_typed_as_missing = po_as_typed.str.strip().ne('') & invoices['po_number'].isna()
po_reformatted = po_raw.notna() & invoices['po_number'].notna() & (po_raw != invoices['po_number'])
non_iso_invoice_date = invoices_raw['invoice_date'].notna() & ~invoices_raw['invoice_date'].str.match(iso_pattern)

repairs = [
    # (table, field, issue, rows changed, raw values, rule applied)
    *[('invoices', column, 'Inconsistent casing or stray spaces', invoices_raw[column] != invoices[column],
       invoices_raw[column], 'Mapped to the canonical value')
      for column in ['department', 'category', 'status']],
    ('invoices', 'invoice_date', 'Date not in ISO format (YYYY-MM-DD)', non_iso_invoice_date,
     invoices_raw['invoice_date'], 'Parsed with a list of known formats'),
    ('invoices', 'po_number', 'Missing PO typed as text', po_typed_as_missing,
     po_as_typed, 'Read as no PO (maverick spend)'),
    ('invoices', 'po_number', 'PO number with spaces or a prefix', po_reformatted,
     po_raw, 'Kept the 10-digit number starting with 45'),
    ('vendors', 'vendor_name', 'Stray spaces', vendors_raw['vendor_name'] != vendors['vendor_name'],
     vendors_raw['vendor_name'], 'Trimmed and collapsed spaces'),
    ('vendors', 'payment_terms', 'Non-standard spelling', vendors_raw['payment_terms'] != vendors['payment_terms'],
     vendors_raw['payment_terms'], 'Parsed into discount %, discount days and net days'),
]
cleaning_log = pd.DataFrame([
    {'table': table, 'field': field, 'issue': issue, 'rows': int(changed.sum()),
     'examples': raw_examples(raw, changed), 'rule': rule}
    for table, field, issue, changed, raw, rule in repairs
])
assert (cleaning_log['rows'] > 0).all(), 'a repair that touches no rows should be removed from the log'
cleaning_log

,table,field,issue,rows,examples,rule
0,invoices,department,Inconsistent casing or stray spaces,305,' Legal ' · 'Hr' · 'MARKETING',Mapped to the canonical value
1,invoices,category,Inconsistent casing or stray spaces,300,' Consulting ' · ' Professional Services '...,Mapped to the canonical value
2,invoices,status,Inconsistent casing or stray spaces,215,' Paid ' · 'PAID' · 'paid',Mapped to the canonical value
3,invoices,invoice_date,Date not in ISO format (YYYY-MM-DD),202,'2025/09/10' · '01/01/2026' · '05-Feb-2026',Parsed with a list of known formats
4,invoices,po_number,Missing PO typed as text,410,'-' · 'n/a' · 'N/A',Read as no PO (maverick spend)
5,invoices,po_number,PO number with spaces or a prefix,267,' 4500520065 ' · 'PO 4502760099',Kept the 10-digit number starting with 45
6,vendors,vendor_name,Stray spaces,3,"' Mcclure, Ward and Lee ' · 'Doyle Ltd' · '...",Trimmed and collapsed spaces
7,vendors,payment_terms,Non-standard spelling,7,'NET14' · 'net 14' · ' Net 30 ',"Parsed into discount %, discount days and net ..."


## Join clean vendor names and write output

Attach the cleaned `vendor_name` for display purposes (KPI/anomaly logic keys off `vendor_id`, never the name). `budgets.csv` was never messified at generation time, so it's copied through unchanged for consistency — stage 3 should only ever read from `data/processed/`. The validated event log is written as `invoice_events.parquet`. The cleaning log is written as `cleaning_log.parquet`.


In [14]:
invoices_clean = invoices.merge(vendors[['vendor_id', 'vendor_name']], on='vendor_id', how='left')

invoices_clean.to_parquet(PROCESSED_DIR / 'invoices_clean.parquet', index=False)
vendors.to_parquet(PROCESSED_DIR / 'vendors_clean.parquet', index=False)
budgets.to_parquet(PROCESSED_DIR / 'budgets.parquet', index=False)
events.to_parquet(PROCESSED_DIR / 'invoice_events.parquet', index=False)
cleaning_log.to_parquet(PROCESSED_DIR / 'cleaning_log.parquet', index=False)

print(f'wrote {len(invoices_clean)} cleaned invoice rows to {PROCESSED_DIR}')
invoices_clean.head(3)

wrote 2550 cleaned invoice rows to ../data/processed


,invoice_id,vendor_id,po_number,department,category,invoice_date,submitted_date,approved_date,paid_date,amount,currency,status,has_po,missing_submitted_date,processing_days,extract_date,payment_terms,discount_pct,discount_days,net_days,due_date,discount_date,vendor_name
0,INV00001,V020,NaN,Legal,Software & IT Services,2026-01-19,2026-01-23,2026-01-26,2026-01-29,2122.75,EUR,Paid,False,False,6.0,2026-08-31,Net 30,0.0,<NA>,30,2026-02-18,NaT,"Reid, Ferguson and Sanchez"
1,INV00002,V003,4560641662,Operations,Consulting,2026-07-17,2026-07-18,2026-07-22,2026-07-23,760.84,EUR,Paid,True,False,5.0,2026-08-31,Net 30,0.0,<NA>,30,2026-08-16,NaT,"Mcclain, Miller and Henderson"
2,INV00003,V016,4573723628,Marketing,Travel,2026-04-13,2026-04-13,2026-04-21,2026-04-26,19385.80,EUR,Paid,True,False,13.0,2026-08-31,2/10 Net 30,2.0,10,30,2026-05-13,2026-04-23,Galloway-Wyatt


## Databricks only: register Delta tables in Unity Catalog

Parquet files in a folder are just files; a **Delta table** in Unity Catalog is a named, governed table (`workspace.finance_ops.invoices_clean`) that anyone in the workspace can find in the Catalog Explorer and query with SQL — with ACID writes and version history ("time travel") on top of Parquet.

`spark.createDataFrame(pandas_df)` is used purely to hand the already-cleaned pandas data to the catalog. Skipped automatically when running locally.

In [15]:
if ON_DATABRICKS:
    tables = {
        'invoices_clean': invoices_clean,
        'vendors_clean': vendors,
        'budgets': budgets,
        'invoice_events': events,
        'cleaning_log': cleaning_log,
    }
    for table_name, df in tables.items():
        (spark.createDataFrame(df)
            .write.mode('overwrite')
            .option('overwriteSchema', 'true')
            .saveAsTable(f'{CATALOG_SCHEMA}.{table_name}'))
        print(f'wrote Delta table {CATALOG_SCHEMA}.{table_name} ({len(df)} rows)')
else:
    print('local run: skipping Delta table registration (parquet output above is all stage 3 needs)')

local run: skipping Delta table registration (parquet output above is all stage 3 needs)


## Databricks only: cross-check with Databricks SQL

Query the Delta table with SQL and assert it agrees with pandas — spend per department must be identical whichever engine computes it. This is a validation step, not part of the pipeline: stage 3 (`src/kpis.py`) still reads the local parquet output.

In [16]:
if ON_DATABRICKS:
    spend_sql = spark.sql(f'''
        SELECT department,
               COUNT(*)              AS invoices,
               ROUND(SUM(amount), 2) AS total_spend
        FROM {CATALOG_SCHEMA}.invoices_clean
        GROUP BY department
        ORDER BY total_spend DESC
    ''')
    display(spend_sql)

    sql_totals = spend_sql.toPandas().set_index('department')['total_spend'].sort_index()
    pandas_totals = invoices_clean.groupby('department')['amount'].sum().round(2).sort_index()
    assert (sql_totals - pandas_totals).abs().max() < 0.01, 'Databricks SQL and pandas disagree'
    print('Databricks SQL totals match pandas for every department')
else:
    print('local run: skipping Databricks SQL cross-check')

local run: skipping Databricks SQL cross-check


## Cleaning summary

For the README's honest-limitations section.

In [17]:
print('Cleaning summary')
print('-----------------')
print(f"rows in / out:              {len(invoices_raw)} / {len(invoices_clean)}")
print(f"invoice_date reparsed:      {non_iso_dates} rows had a non-ISO source format")
print(f"rows missing submitted_date:{int(invoices['missing_submitted_date'].sum())} (kept, excluded from processing-time KPI only)")
print(f"vendor names whitespace-fixed: {(vendors_raw['vendor_name'] != vendors['vendor_name']).sum()}")
print(f"payment terms re-typed:     {(vendors_raw['payment_terms'] != vendors['payment_terms']).sum()} vendors")
print(f"invoices without a PO:      {int((~invoices_clean['has_po']).sum())} (maverick spend)")
print(f"open at extract {EXTRACT_DATE.date()}: {int(invoices_clean['paid_date'].isna().sum())}")
print(f"events validated:           {len(events)}")

Cleaning summary
-----------------
rows in / out:              2550 / 2550
invoice_date reparsed:      202 rows had a non-ISO source format
rows missing submitted_date:24 (kept, excluded from processing-time KPI only)
vendor names whitespace-fixed: 3
payment terms re-typed:     7 vendors
invoices without a PO:      517 (maverick spend)
open at extract 2026-08-31: 93
events validated:           10882
